# 4-2: Network Analysis in Python

In this tutorial, we'll cover how to study a character network from Victor Hugo's _Les Misérables_ using Python.

The goal is not to learn every network method. The goal is to understand a basic workflow, like this:

1. Understanding nodes and edges
2. Building a network graph
3. Calculating simple network measures
4. Visualizing the network
5. Interpreting the results carefully as digital humanists

By the end, you should be able to explain what a node, edge, weight, degree, centrality, and community mean in a network analysis project and understand a network analysis workflow in Python.

## Network Analysis: An Overview

Simply put, a __network__ is a way of representing relationships. These relationships are often quite explicit––things like shared followers on social media or mutual appearances of characters in a scene. Not all networks are that obvious, though. Much depends on what you define as the parts of the network.

In that regard, it's important to note that networks are made of two basic parts:

- __Nodes:__ the things in the network. These are sometimes called vertices.
- __Edges:__ the relationships between those things. These are sometimes called links or ties.

In humanities research, nodes might be people, places, books, newspapers, letters, manuscripts, or concepts. Edges might represent correspondence, co-publication, citation, trade, shared locations, shared vocabulary, or co-appearance in a text. What you define as the nodes and edges becomes extraordinarily important for DH network analysis. Nothing can come of network analysis unless your nodes and edges are clearly defined.

A network can also have extra information:

- __Node attributes:__ information about nodes, such as a character's name or description.
- __Edge attributes:__ information about edges, such as the strength of a relationship.
- __Weights:__ numbers that tell us how strong or frequent an edge is.
- __Direction:__ whether a relationship points one way, such as A cites B, or has no direction, such as A and B appear in the same chapter.

Network analysis is often useful in DH because it helps us move between close reading and distant reading. It can show patterns that would be otherwise hard to see. It can also provide more evidence for arguments about particular connections between subject matter––arguments that prove or refute how aspects of humanities data are bound together.

### Important Interpretive Caution

Network metrics are not neutral facts about a text. They are measurements of a model that we created.

For example, in this lesson on Les Mis, an edge means that two characters co-appear in the same chapter. That does __not__ necessarily mean they are friends, speak to each other, or have equal narrative importance.

A good DH network analysis always asks:

- What exactly does an edge mean?
- What does the data leave out?
- Which characters or relationships might be overemphasized by this model?
- How can we connect the network results back to interpretation?

To make any knowledge claims or arguments about our subject matter, we would need to compare our network analyses findings to the text itself. In DH research, this is particularly important. In other words, if you were to try to argue anything from the following lesson, you should also read Les Mis!

### The _Les Misérables_ Character Network

This tutorial uses character network data from the GitHub repository [MADStudioNU/lesmiserables-character-network](https://github.com/MADStudioNU/lesmiserables-character-network).

The repository describes the data as an update to Donald Knuth's original _Les Misérables_ character network from the Stanford GraphBase. The updated dataset was prepared by Professor Michal Peled Ginsburg at Northwestern University, with support from the WCAS Multimedia Learning Center.

We'll import two CSV files directly from GitHub:

- [`jean-complete-node.csv`](https://github.com/MADStudioNU/lesmiserables-character-network/blob/master/parsed_data/jean-complete-node.csv): one row per character.
- [`jean-complete-edge.csv`](https://github.com/MADStudioNU/lesmiserables-character-network/blob/master/parsed_data/jean-complete-edge.csv): one row per character co-appearance.

This is useful beyond today's lesson because many humanities datasets are shared through GitHub. To load a GitHub CSV directly with `pandas`, we use the __raw__ file URL, not the regular GitHub preview page.

Whenever you do this, you should also check the repository's licensing agreement and give credit where credit is due. In this case:

__Credit:__ Data adapted from MADStudioNU, "Les Miserables Character Network," based on Donald Knuth's Stanford GraphBase `jean.dat`. Scholarly updates by Michal Peled Ginsburg, Northwestern University. Licensed under [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).

Also, here's a brief overview of the text in question:

Les Misérables is Victor Hugo’s 1862 novel about poverty, justice, love, and political struggle in nineteenth-century France. Its central figure is Jean Valjean, a former prisoner who tries to rebuild his life while being pursued by the police inspector Javert. Around Valjean, the novel follows many interconnected characters, including Fantine, Cosette, Marius, the Thénardiers, and student revolutionaries in Paris. Because the novel moves across families, neighborhoods, institutions, and political movements, it is especially well suited to studying character networks.

## Setup

We'll use four Python libraries:

- `pandas`
- `matplotlib` and `seaborn`
- `networkx` to build and analyze networks

The new one for us is `networkx`. This is a popular Python library for working with network data. It has a whole long list of methods and functions that make it easy to assess networks. To learn more about it, check out the [documentation here.](https://networkx.org/en/)

We'll use `%pip install networkx` to install the library. This is a Jupyter magic command that installs a Python package into the environment connected to this notebook.

If `networkx` is already installed, the command should simply tell you that the requirement is already satisfied:

In [ ]:
%pip install networkx

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

## Step 1: Load The Data From GitHub

We have two tables:

- `nodes`: information about characters
- `edges`: information about co-appearances between characters

Instead of using local files from our course repository, we'll load the CSV files directly from GitHub.

Notice that these URLs begin with `https://raw.githubusercontent.com/`. That means the link points to the raw CSV data, not the GitHub page that previews the file in a browser.


In [ ]:
nodes_url = "https://raw.githubusercontent.com/MADStudioNU/lesmiserables-character-network/master/parsed_data/jean-complete-node.csv"
edges_url = "https://raw.githubusercontent.com/MADStudioNU/lesmiserables-character-network/master/parsed_data/jean-complete-edge.csv"

# w/ handles for na values
nodes = pd.read_csv(nodes_url, dtype=str, keep_default_na=False)
edges = pd.read_csv(edges_url, dtype=str, keep_default_na=False)

Let's preview the character table.


In [ ]:
nodes.head()

Each row in `nodes` is a character. The `Id` column gives a short character code. The `Label` column gives a readable name.

Let's look at our edges, too:

In [ ]:
edges.head()

Each row in `edges` is a co-appearance between two character IDs––in other words, instances in the novel when characters appear together in a chapter. The `Label` column stores the chapter where that co-appearance happens.

With these details in mind, what sorts of possibilities and limitations come to mind when we start thinking about this network data and the kinds of information we can glean from it?

In [ ]:
print(f"Number of character rows: {len(nodes)}")
print(f"Number of raw co-appearance rows: {len(edges)}")
print(f"Number of chapters represented: {edges['Label'].nunique()}")

## Step 2: Understand Raw Edges And Weighted Edges

Our `edges` data describes what are sometimes called __raw edges__. That means for each connection between characters, the table has a single row representing the instance of connection (a single raw edge). This also means the data has repeated pairs. That's because the same pair of characters may co-appear in multiple chapters.

For many network analyses, we want one row per pair of characters and a `weight` column that counts how many times the pair appears. This gives us the __weighted edge__ between network nodes. Why? Well, imagine trying to visualize a network where each connection––each raw edge––is drawn with a single line. That would result in many, many lines drawn between points in the network. Rather than cluttering our connections, we could just maintain single lines in the visualization, but lines that are wider or heavier based on their __weighted edge__ value.

To do this, we need to calculate the weighted edges and create a new table that would look something like this:

| source | target | weight |
| --- | --- | --- |
| JV | CO | 62 |

This would mean that Jean Valjean and Cosette co-appear 62 times in the data.

Also, it's important to note that this is an example of an __undirected network__––a network where the direction of the connections doesn't matter. __Directed networks__, on the other hand, are those where the source only connects to the target, not the other way around. An example of a directed network would be flight patterns between airports. The airports would be the nodes, the flight patterns the edges. In that sort of case, it matters which direction the flights are going, so you would need to account for that in any analyses of the network. In our case, the nodes are the chapters where characters appear together. It doesn't matter who appears in the scenes first––it just matters that they're there together, so it's an example of an undirected network.

But back to weighted edges. To calculate the weighted edges, we'll create a new dataframe called `weighted_edges` by running some transformations on our edges data. In particular, we'll use a series of methods to do the following:

1. `.groupby` to group the table by `source` and `target`
2. `.size()` to count the number of rows in each group
3. `.reset_index()` to turn the counts into a column called `weight`
4. and `.sort_values()` to sort the table so the strongest co-appearances come first

In [ ]:
weighted_edges = (
    edges
    .groupby(["Source", "Target"])
    .size()
    .reset_index(name="weight")
    .sort_values("weight", ascending=False)
)

weighted_edges.head(10)

Excellent! Using this `weighted_edges` table, we're now ready to analyze our network.

But first, let's compare the number raw edges versus weighted edges in our network:

In [ ]:
print(f"Raw edge rows: {len(edges)}")
print(f"Weighted character-pair edges: {len(weighted_edges)}")

Can we summarize the difference between these ways of understanding edges? What do each tell us about the character co-appearances in Les Mis?

## Step 3: Build A Network With `networkx`

`networkx` allows us to represent our network data as a graph object. We can do this with the simple method `.Graph()`. Because ours is an undirected network, `.Graph()` is the proper object method. If we were working with a directional network, we'd use `.DiGraph()`. If we were working with multiple types of edges, we'd use `.MultiGraph()`.

We'll start by assigning an empty graph object to `G`. Then we can add our character nodes and start building this graph object.

In [ ]:
G = nx.Graph()

In [ ]:
type(G)

Then we add our nodes. For each row in `nodes`, we add one node to the graph. We also attach the character's readable label as node attributes. This all can be done with a for-loop and the `.iterrows()` method:

In [ ]:
for index, row in nodes.iterrows():
    character_id = row["Id"]
    character_label = row["Label"]
    
    G.add_node(
        character_id,
        label=character_label
    )

# notice how networkx allows you to print the G object w/ a description
print(G)

Now we add the weighted edges.

Each row in `weighted_edges` becomes one edge in the graph. Once again, we'll iterate but this time over our `weighted_edges` dataframe, saving the Source and Target values as parts of our G object.

In [ ]:
for index, row in weighted_edges.iterrows():
    source = row["Source"]
    target = row["Target"]
    # be sure to read weight as int
    weight = int(row["weight"])
    
    G.add_edge(source, target, weight=weight)

print(G)

One thing about `networkx`: its graph objects are hard to view in tabular form. This is because the `G` object actually its two components––the nodes and the edges––in separate structures. 

That's why when you try to print `G`, you only get the description:

In [ ]:
print(G)

But if you print the sparate components, you start to see what `G` actually contains:

In [ ]:
print(G.nodes)
print(type(G.nodes))
print(len(G.nodes))

In [ ]:
print(G.edges)
print(type(G.edges))
print(len(G.edges))

`networkx` also has methods for its Graph object components. Notice how `.number_of_nodes()` and `.number_of_edges()` reveals the same info as using `len()`:

In [ ]:
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

We can also inspect one node by its character ID. `JV`, for example, is Jean Valjean:

In [ ]:
G.nodes["JV"]

We can also inspect one edge. The edge between `JV` and `CO` connects Jean Valjean and Cosette. When we call it, we get its weight:

In [ ]:
G.edges["JV", "CO"]

Interacting with these Graph objects is sort of like feeling around in the dark. You can look at the constituent parts, but unlike Pandas dataframes or other data structures, `networkx` doesn't provide singular views of their data structures. That's just something to keep in mind while using this library.

## Step 4: Basic Network Summary

Okay, now let's calculate a few whole-network measures. In particular, we'll check to see if the network is fully __connected__, its __density__, and its __average degree__ of connection. What do these measure mean?

- __Connected:__ every character can be reached from every other character through some path. A fully connected network means there's no isolated clusters or nodes. Everything is some degree of separation away from everything else.
- __Density:__ how many possible edges actually exist. A density near 0 means a sparse network. Higher density means a a more interconnected network. A density of 1 means every node has a connection to every other node––a fully connected network.
- __Average degree:__ the average number of neighbors per node––take the edge connections between nodes and divide them by the total number of connected nodes. This gives you the average degree of connectedness.

In terms of our character network, what could these measurements tell us?

Here's how to use `networkx` to quickly calculate these network measures:


In [ ]:
is_connected = nx.is_connected(G)
density = nx.density(G)
average_degree = sum(dict(G.degree()).values()) / G.number_of_nodes()

print(f"Is the network connected? {is_connected}")
print(f"Density: {density:.4f}")
print(f"Average degree: {average_degree:.2f}")

## Step 5: Degree Centrality

A character's __degree__ is the number of other characters they are connected to (i.e., how many other characters they appear in chapters with). This does not count how often they co-appear. It only counts how many different characters they co-appear with.

To assess each character's degree, we can use the `.degree()` method and read the results as a dictionary, like this:

In [ ]:
degree_dict = dict(G.degree())
degree_dict

Notice how the keys are character IDs and the values are degrees (no. of other characters they appear with). Let's convert the dictionary into a DataFrame to review this data more easily:

In [ ]:
degree_df = pd.DataFrame({
    "Id": degree_dict.keys(),
    "degree": degree_dict.values()
})

degree_df.head()

That's better, but it'd be even better still if we had the full character names alongside this degree and IDs. We can add that easily by merging this dataframe with our nodes data, like this:

In [ ]:
# the merge function combines nodes dataframe w/ degree_df on value "ID"
degree_df = degree_df.merge(nodes, on="Id")
# sort ascending False
degree_df = degree_df.sort_values("degree", ascending=False)

degree_df.head(10)

Hmmm, so looks like Jean Valjean has the highest degree of all the characters in Les Mis. What does that tell you about the work?

## Step 6: Weighted Degree

Degree asks: how many different characters is this character connected to?

__Weighted degree__, on the other hand, asks: how strong are this character's connections overall?

Here, we need to look at those edge weights we calculated earlier (i.e., the number of co-appearances). To do that, we'll convert the weights to a dictionary, like this:

In [ ]:
weighted_degree_dict = dict(G.degree(weight="weight"))

Then, for ease of review, we'll use merge and sort_values again to look at this data as a dataframe, like this:

In [ ]:
weighted_degree_df = pd.DataFrame({
    "Id": weighted_degree_dict.keys(),
    "weighted_degree": weighted_degree_dict.values()
})

weighted_degree_df = weighted_degree_df.merge(nodes, on="Id")
weighted_degree_df = weighted_degree_df.sort_values("weighted_degree", ascending=False)

weighted_degree_df.head(10)

What do these weighted degrees tell you about the characters and the work overall?

A character can have high degree because they appear with many characters. But a character can have high weighted degree because they repeatedly appear with the same characters. What could that tell you about the characters and their connections to one another?

## Step 7: Betweenness Centrality

__Betweenness centrality__ is a measure of the "distance" between connections in a network. It asks whether a node often sits on the shortest paths between other nodes.

In our case, this measure can help us find potential bridges between different parts of a network––the bridges between characters who would otherwise not be connected.

`networkx` makes calculating this value easy with the `.betweenness_centrality()` method:

In [ ]:
betweenness_dict = nx.betweenness_centrality(G)
betweenness_dict

Notice how many characters have a betweenness centrality of 0. What does this mean?

It means they never serve as the connecting node between two other characters. Betweenness centrality, in other words, is asking “how often does this character help connect other characters to each other?”

Let's make this measurement more readable as a sorted dataframe:

In [ ]:
betweenness_df = pd.DataFrame({
    "Id": betweenness_dict.keys(),
    "betweenness": betweenness_dict.values()
})

betweenness_df = betweenness_df.merge(nodes, on="Id")
betweenness_df = betweenness_df.sort_values("betweenness", ascending=False)

betweenness_df.head(10)

What could this measurement tell you about the characters with the highest betweenness centrality?

## Step 8: Combine The Metrics

It is often useful to put several network measures into one table. Let's do that by taking our various dictionaries from the above metrics (`degree_dict`, `weighted_degree_dict`, and `betweenness_dict`) and combining them into a dataframe with the `.map()` method:

In [ ]:
centrality_df = nodes.copy()

centrality_df["degree"] = centrality_df["Id"].map(degree_dict)
centrality_df["weighted_degree"] = centrality_df["Id"].map(weighted_degree_dict)
centrality_df["betweenness"] = centrality_df["Id"].map(betweenness_dict)

# sort by degree to start
centrality_df = centrality_df.sort_values("degree", ascending=False)
centrality_df.head(10)

Now we can compare metrics side by side.

Ask yourself:

- Which characters are high across all measures?
- Which characters are high in one measure but not another?
- What might explain those differences?


## Step 9: Communities

A **community** is a group of nodes that are more densely connected to each other than to the rest of the network.

Community detection is exploratory. It can suggest clusters, but it doesn't prove that those clusters are socially or narratively meaningful. To understand them, you have to both identify the communities AND interpret them.

To identify them, we'll use the `.community` and `.greedy_modularity_communities` methods. This algorithm looks for groups of nodes where there are many connections inside the group and fewer connections going outside the group. We can implement it like this:

In [ ]:
communities = nx.community.greedy_modularity_communities(G, weight="weight")

print(f"Number of communities detected: {len(communities)}")

The result is a list of communities. Each community is a set of character IDs:

In [ ]:
communities[0]

Let's create a dictionary that maps each character ID to a community number. That way, we can see which community they're part of:

In [ ]:
community_lookup = {}

for community_number, community in enumerate(communities, start=1):
    for character_id in community:
        community_lookup[character_id] = community_number

print(community_lookup)

Now we add the community number to our centrality table with the `.map` method again:

In [ ]:
centrality_df["community"] = centrality_df["Id"].map(community_lookup)
centrality_df.head(10)

How large are the communities?


In [ ]:
centrality_df["community"].value_counts().sort_index()

Let's see the top characters in each community by weighted degree.


In [ ]:
top_by_community = (
    centrality_df
    .sort_values(["community", "weighted_degree"], ascending=[True, False])
    .groupby("community")
    .head(3)
)

top_by_community[["community", "Label", "degree", "weighted_degree", "betweenness"]]

What do these communities tell us? Do you notice anything interesting about them? If you were to read Les Mis with these communities in mind, how would this exploratory method inform your reading of the work?

## Step 10: Visualize Top Characters

Before drawing the whole network, let's make a simple bar chart. This is often easier to read than a network visualization.

Let's look at the top ten characters by degree. To subset this data, we'll use `.sort_values` and `.head`:

In [ ]:
top_degree = centrality_df.sort_values("degree", ascending=False).head(10)

top_degree[["Label", "degree", "weighted_degree", "community"]]

Now let's build our barchart. We'll use plt and sns here.

In [ ]:
# This makes some of our plots look a little cleaner
sns.set_theme(style="whitegrid")

top_degree_plot = top_degree.copy()
top_degree_plot["community"] = top_degree_plot["community"].astype(str)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_degree_plot,
    x="degree",
    y="Label",
    hue="community",
    dodge=False,
    palette="tab10"
)

plt.title("Top 10 Characters by Degree")
plt.xlabel("Degree: number of connected characters")
plt.ylabel("Character")
plt.legend(title="Community")
plt.show()

## Step 11: Draw A Manageable Subgraph

Full network visualizations can become messy. A common strategy is to draw a smaller subgraph first.

Here we'll draw the 25 characters with the highest degree. Let's first get that subset of the network:

In [ ]:
top_25_ids = centrality_df.sort_values("degree", ascending=False).head(25)["Id"].tolist()

top_25_graph = G.subgraph(top_25_ids).copy()

print(top_25_graph)

Now we need to decide how to present this part of the network. We'll use a layout that decides decides where nodes appear on the screen. `networkx` makes this easy with its `.spring_layout` method. This is an algorithmic approach to the layout that places connected nodes near each other. The `seed` makes the layout repeatable, so node works with the same visualization. 

The seed can be set to whatever you want, but if you want to seem like a culturally knowledgeable programmer, you'll set it to 42. This is an old inside joke among programmers. Any time you have to set a random number, set it to 42––the secret of the universe! The answer to everything! (Read Hitchhiker's Guide to the Galaxy and you'll understand the joke).

In [ ]:
top_25_layout = nx.spring_layout(top_25_graph, seed=42, weight="weight")

We also need to size nodes by degree and color them by community. To do that, we'll iteratre over them and create lists of visual settings, one value per node:

In [ ]:
centrality_by_id = centrality_df.set_index("Id")

node_sizes = []
node_colors = []

for node in top_25_graph.nodes():
    degree = centrality_by_id.loc[node, "degree"]
    community = centrality_by_id.loc[node, "community"]
    
    node_sizes.append(degree * 80)
    node_colors.append(community)

Let's also make stronger edges slightly thicker:

In [ ]:
edge_widths = []

for source, target in top_25_graph.edges():
    weight = top_25_graph.edges[source, target]["weight"]
    edge_widths.append(weight * 0.15)

Now we draw the subgraph. Remember: the graph must have nodes, edges, and labels to make it human-readable. With our above metrics, we can plug these values into the standard `networkx` figure components:

In [ ]:
plt.figure(figsize=(12, 10))

nx.draw_networkx_edges(
    top_25_graph,
    top_25_layout,
    width=edge_widths,
    alpha=0.35,
    edge_color="black"
)

nx.draw_networkx_nodes(
    top_25_graph,
    top_25_layout,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.tab10,
    alpha=0.9
)

labels = nx.get_node_attributes(top_25_graph, "label")

nx.draw_networkx_labels(
    top_25_graph,
    top_25_layout,
    labels=labels,
    font_size=9
)

plt.title("Les Misérables Character Network: Top 25 Characters by Degree")
plt.axis("off")
plt.show()

## Step 12: Draw The Full Network

Now that we know the basic method, we can draw the full network.

This plot is more complex, so we'll label only the top 15 characters by degree. Otherwise, all those labels will appear atop each other, becoming unreadable!

In [ ]:
# k adds some separation between nodes
full_layout = nx.spring_layout(G, seed=42, weight="weight", k=0.4)

In [ ]:
node_sizes = []
node_colors = []

for node in G.nodes():
    degree = centrality_by_id.loc[node, "degree"]
    community = centrality_by_id.loc[node, "community"]
    
    node_sizes.append(degree * 18 + 20)
    node_colors.append(community)

print(node_sizes)
print(node_colors)

In [ ]:
edge_widths = []

for source, target in G.edges():
    weight = G.edges[source, target]["weight"]
    edge_widths.append(weight * 0.04)

print(edge_widths)

In [ ]:
top_15_ids = centrality_df.sort_values("degree", ascending=False).head(15)["Id"]

full_labels = {}

for character_id in top_15_ids:
    full_labels[character_id] = G.nodes[character_id]["label"]

print(full_labels)

In [ ]:
plt.figure(figsize=(13, 13))

nx.draw_networkx_edges(
    G,
    full_layout,
    width=edge_widths,
    alpha=1,
    edge_color="black"
)

nx.draw_networkx_nodes(
    G,
    full_layout,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.tab10,
    alpha=0.85
)

nx.draw_networkx_labels(
    G,
    full_layout,
    labels=full_labels,
    font_size=9
)

plt.title("Full Les Misérables Character Co-appearance Network")
plt.axis("off")
plt.show()